# Fusion Design Lab — GRPO Training

Train an LLM to optimize stellarator fusion reactor designs using **GRPO** (Group Relative Policy Optimization) with **Unsloth** and **TRL**.

The agent interacts with a constrained optimization environment where it adjusts 4 geometric knobs of a stellarator boundary, aiming to **minimize max elongation** while satisfying 3 hard physics constraints:
- `aspect_ratio ≤ 4.0`
- `average_triangularity ≤ -0.5`
- `edge_iota_over_nfp ≥ 0.3`

Each episode has **6 evaluations** budgeted. The agent produces a plan of actions and the environment scores it via the `constellaration` physics verifier.

**Environment deployed at**: https://creativeengineer-fusion-design-lab.hf.space

**Runtime**: Select GPU (T4 or better) via `Runtime > Change runtime type`.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth vllm
!pip install --no-deps trl
!pip install constellaration openenv-core[core] pydantic fastapi uvicorn
!pip install matplotlib

## 2. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen3-0.6B"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    use_gradient_checkpointing="unsloth",
)

print(f"Model loaded: {MODEL_NAME}")

## 3. Setup Stellarator Environment

We install the environment package directly from the HF Space repository so training runs locally (no network latency). The same environment is deployed at the HF Space URL above.

In [ ]:
%%capture
!pip install git+https://huggingface.co/spaces/CreativeEngineer/fusion-design-lab

In [ ]:
import json
import re
from typing import Final

from fusion_lab.models import StellaratorAction, StellaratorObservation
from server.contract import RESET_SEEDS
from server.environment import BUDGET, StellaratorEnvironment

AVAILABLE_ACTIONS: Final[list[dict[str, str]]] = [
    {"intent": "run", "parameter": p, "direction": d, "magnitude": m}
    for p in ["aspect_ratio", "elongation", "rotational_transform", "triangularity_scale"]
    for d in ["increase", "decrease"]
    for m in ["small", "medium", "large"]
] + [
    {"intent": "restore_best"},
    {"intent": "submit"},
]

ACTION_LABELS: Final[list[str]] = [
    f"{a['intent']} {a.get('parameter', '')} {a.get('direction', '')} {a.get('magnitude', '')}".strip()
    for a in AVAILABLE_ACTIONS
]

# Quick smoke test
env = StellaratorEnvironment()
obs = env.reset(seed=0)
print(
    f"Environment ready. Initial score: {obs.p1_score:.4f}, feasibility: {obs.p1_feasibility:.4f}"
)
print(f"Budget: {obs.budget_remaining}, Constraints satisfied: {obs.constraints_satisfied}")

## 4. Prompt Template & Action Parsing

Each training sample is a prompt describing the stellarator task and initial state. The model generates a plan of actions to optimize the design.

In [ ]:
SYSTEM_PROMPT: Final[
    str
] = """You are an expert stellarator fusion reactor designer. Your goal is to optimize a stellarator design by adjusting 4 geometric parameters to minimize max elongation while satisfying physics constraints.

Constraints:
- aspect_ratio <= 4.0
- average_triangularity <= -0.5
- edge_iota_over_nfp >= 0.3

Available parameters: aspect_ratio, elongation, rotational_transform, triangularity_scale
Available directions: increase, decrease
Available magnitudes: small, medium, large

You have a budget of 6 evaluations. Output a plan of actions as a JSON array. Each action is an object with keys: intent, parameter, direction, magnitude. The last action should be {"intent": "submit"} to finalize your design.

Example:
[{"intent":"run","parameter":"triangularity_scale","direction":"increase","magnitude":"small"},{"intent":"run","parameter":"rotational_transform","direction":"increase","magnitude":"medium"},{"intent":"submit"}]"""


def format_observation(obs: StellaratorObservation) -> str:
    return (
        f"Current stellarator state:\n"
        f"  max_elongation: {obs.max_elongation:.4f}\n"
        f"  aspect_ratio: {obs.aspect_ratio:.4f} (constraint: <= 4.0)\n"
        f"  average_triangularity: {obs.average_triangularity:.6f} (constraint: <= -0.5)\n"
        f"  edge_iota_over_nfp: {obs.edge_iota_over_nfp:.4f} (constraint: >= 0.3)\n"
        f"  p1_score: {obs.p1_score:.4f}\n"
        f"  feasibility: {obs.p1_feasibility:.4f}\n"
        f"  constraints_satisfied: {obs.constraints_satisfied}\n"
        f"  budget_remaining: {obs.budget_remaining}\n"
        f"\nGenerate an action plan as a JSON array to optimize this design."
    )


def build_prompt(obs: StellaratorObservation) -> str:
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{format_observation(obs)}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


def parse_action_plan(text: str) -> list[StellaratorAction]:
    """Parse a JSON action plan from model output."""
    # Find JSON array in the text
    match = re.search(r"\[.*?\]", text, re.DOTALL)
    if not match:
        return []
    try:
        raw = json.loads(match.group())
    except json.JSONDecodeError:
        return []
    actions = []
    for item in raw:
        if not isinstance(item, dict) or "intent" not in item:
            continue
        intent = item["intent"]
        if intent == "submit":
            actions.append(StellaratorAction(intent="submit"))
            break
        if intent == "restore_best":
            actions.append(StellaratorAction(intent="restore_best"))
            continue
        if intent == "run":
            p = item.get("parameter", "")
            d = item.get("direction", "")
            m = item.get("magnitude", "small")
            if p in (
                "aspect_ratio",
                "elongation",
                "rotational_transform",
                "triangularity_scale",
            ) and d in ("increase", "decrease"):
                if m not in ("small", "medium", "large"):
                    m = "small"
                actions.append(
                    StellaratorAction(intent="run", parameter=p, direction=d, magnitude=m)
                )
    return actions


# Test prompt
env = StellaratorEnvironment()
obs = env.reset(seed=0)
prompt = build_prompt(obs)
print(prompt[:500])
print("...")

## 5. Training Dataset

Create prompts from all 3 reset seeds. Each prompt is an initial observation that the model must optimize.

In [ ]:
from datasets import Dataset

prompts = []
for seed_idx in range(len(RESET_SEEDS)):
    env = StellaratorEnvironment()
    obs = env.reset(seed=seed_idx)
    prompt = build_prompt(obs)
    # Repeat each seed to create a larger training set
    for _ in range(50):
        prompts.append({"prompt": prompt, "seed_idx": seed_idx})

dataset = Dataset.from_list(prompts)
dataset = dataset.shuffle(seed=42)
print(f"Training dataset: {len(dataset)} samples from {len(RESET_SEEDS)} seeds")

## 6. Reward Functions

Two reward signals:
1. **Format reward**: Does the completion contain a valid JSON action plan?
2. **Environment reward**: Execute the plan in the stellarator environment and return cumulative reward.

In [ ]:
import traceback


def format_reward_fn(completions: list[str], **kwargs) -> list[float]:
    """Reward for producing a valid, parseable action plan."""
    rewards = []
    for completion in completions:
        actions = parse_action_plan(completion)
        if len(actions) == 0:
            rewards.append(-1.0)
        elif any(a.intent == "submit" for a in actions):
            rewards.append(1.0)  # valid plan ending with submit
        else:
            rewards.append(0.0)  # valid actions but no submit
    return rewards


def environment_reward_fn(
    completions: list[str], seed_idx: list[int] | None = None, **kwargs
) -> list[float]:
    """Execute each action plan in the environment and return cumulative reward."""
    rewards = []
    seeds = seed_idx if seed_idx is not None else [0] * len(completions)
    for i, completion in enumerate(completions):
        try:
            actions = parse_action_plan(completion)
            if len(actions) == 0:
                rewards.append(-3.0)
                continue
            env = StellaratorEnvironment()
            env.reset(seed=int(seeds[i]) % len(RESET_SEEDS))
            total_reward = 0.0
            for action in actions[:BUDGET]:
                obs = env.step(action)
                total_reward += float(obs.reward or 0.0)
                if obs.done:
                    break
            rewards.append(total_reward)
        except Exception:
            traceback.print_exc()
            rewards.append(-3.0)
    return rewards


# Test reward functions with a hand-crafted plan
test_plan = json.dumps(
    [
        {
            "intent": "run",
            "parameter": "triangularity_scale",
            "direction": "increase",
            "magnitude": "small",
        },
        {
            "intent": "run",
            "parameter": "rotational_transform",
            "direction": "increase",
            "magnitude": "medium",
        },
        {"intent": "submit"},
    ]
)
print(f"Format reward: {format_reward_fn([test_plan])}")
print(f"Environment reward: {environment_reward_fn([test_plan], seed_idx=[0])}")

## 7. GRPO Training

Train the model using Group Relative Policy Optimization. GRPO generates multiple completions per prompt and updates the policy toward higher-reward completions.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

MAX_PROMPT_LENGTH = 768
MAX_COMPLETION_LENGTH = MAX_SEQ_LENGTH - MAX_PROMPT_LENGTH

training_args = GRPOConfig(
    output_dir="./grpo_fusion_output",
    learning_rate=2e-4,
    num_generations=4,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=60,
    temperature=1.0,
    logging_steps=1,
    save_steps=20,
    bf16=True,
    report_to="none",
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward_fn, environment_reward_fn],
    args=training_args,
    train_dataset=dataset,
)

print("Starting GRPO training...")
train_result = trainer.train()
print(f"Training complete. Total steps: {train_result.global_step}")

## 8. Training Results

Visualize reward improvement over training steps.

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps = [entry["step"] for entry in log_history if "loss" in entry]
losses = [entry["loss"] for entry in log_history if "loss" in entry]

# Extract reward metrics if available
reward_steps = [
    entry["step"]
    for entry in log_history
    if "reward" in entry or "rewards/environment_reward_fn" in entry
]
rewards = [
    entry.get("reward", entry.get("rewards/environment_reward_fn", 0))
    for entry in log_history
    if "reward" in entry or "rewards/environment_reward_fn" in entry
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(steps, losses, "b-", alpha=0.7)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("GRPO Training Loss")
axes[0].grid(True, alpha=0.3)

if rewards:
    axes[1].plot(reward_steps, rewards, "g-o", alpha=0.7, markersize=3)
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Mean Reward")
    axes[1].set_title("Environment Reward Over Training")
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "Reward metrics not logged", ha="center", va="center")

plt.suptitle("Fusion Design Lab — GRPO Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved training_curves.png")

## 9. Evaluate Trained Policy

Generate action plans from the trained model and compare against random baselines.

In [ ]:
import random

FastLanguageModel.for_inference(model)


def run_episode_with_model(seed_idx: int) -> tuple[float, list[str]]:
    """Run one episode using the trained model."""
    env = StellaratorEnvironment()
    obs = env.reset(seed=seed_idx)
    prompt = build_prompt(obs)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_COMPLETION_LENGTH,
        temperature=0.7,
        do_sample=True,
    )
    completion = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    actions = parse_action_plan(completion)
    trace = []
    total_reward = 0.0
    for action in actions[:BUDGET]:
        obs = env.step(action)
        r = float(obs.reward or 0.0)
        total_reward += r
        trace.append(
            f"  {action.intent} {action.parameter or ''} {action.direction or ''} {action.magnitude or ''} → reward={r:.3f} score={obs.p1_score:.4f} feasible={obs.constraints_satisfied}".strip()
        )
        if obs.done:
            break
    return total_reward, trace


def run_random_episode(seed_idx: int) -> float:
    """Run one episode with random actions for comparison."""
    env = StellaratorEnvironment()
    env.reset(seed=seed_idx)
    total_reward = 0.0
    for step in range(BUDGET - 1):
        spec = random.choice(AVAILABLE_ACTIONS[:24])  # run actions only
        action = StellaratorAction(**spec)
        obs = env.step(action)
        total_reward += float(obs.reward or 0.0)
        if obs.done:
            return total_reward
    # submit on last step
    obs = env.step(StellaratorAction(intent="submit"))
    total_reward += float(obs.reward or 0.0)
    return total_reward


# Evaluate
print("=" * 60)
print("TRAINED MODEL EPISODES")
print("=" * 60)
trained_rewards = []
for seed in range(len(RESET_SEEDS)):
    reward, trace = run_episode_with_model(seed)
    trained_rewards.append(reward)
    print(f"\nSeed {seed} — Total reward: {reward:.3f}")
    for line in trace:
        print(f"  {line}")

print(f"\nMean trained reward: {sum(trained_rewards) / len(trained_rewards):.3f}")

print("\n" + "=" * 60)
print("RANDOM BASELINE (10 episodes per seed)")
print("=" * 60)
random_rewards = []
for seed in range(len(RESET_SEEDS)):
    seed_rewards = [run_random_episode(seed) for _ in range(10)]
    random_rewards.extend(seed_rewards)
    print(
        f"Seed {seed} — Mean: {sum(seed_rewards) / len(seed_rewards):.3f}, Best: {max(seed_rewards):.3f}"
    )

print(f"\nMean random reward: {sum(random_rewards) / len(random_rewards):.3f}")
print(f"Mean trained reward: {sum(trained_rewards) / len(trained_rewards):.3f}")

## 10. Connect to Deployed HF Space

Demonstrate connecting to the live environment on Hugging Face Spaces.

In [ ]:
from fusion_lab.client import FusionLabClient
from fusion_lab.models import StellaratorAction

HF_SPACE_URL = "https://creativeengineer-fusion-design-lab.hf.space"

with FusionLabClient(base_url=HF_SPACE_URL).sync() as client:
    obs = client.reset()
    print(f"Connected to HF Space: {HF_SPACE_URL}")
    print("Initial observation:")
    print(f"  max_elongation: {obs.observation.max_elongation:.4f}")
    print(f"  aspect_ratio: {obs.observation.aspect_ratio:.4f}")
    print(f"  p1_score: {obs.observation.p1_score:.4f}")
    print(f"  constraints_satisfied: {obs.observation.constraints_satisfied}")
    print(f"  budget_remaining: {obs.observation.budget_remaining}")

    # Run one action from the trained model
    prompt = build_prompt(obs.observation)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, max_new_tokens=MAX_COMPLETION_LENGTH, temperature=0.7, do_sample=True
    )
    completion = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    actions = parse_action_plan(completion)

    print(f"\nModel generated {len(actions)} actions:")
    for i, action in enumerate(actions[:BUDGET]):
        result = client.step(action)
        print(
            f"  Step {i + 1}: {action.intent} {action.parameter or ''} {action.direction or ''} {action.magnitude or ''} → reward={result.reward:.3f}"
        )
        if result.done:
            print(f"  Episode done. Final score: {result.observation.p1_score:.4f}")
            break

print("\nDone! Environment is live and accessible for training and evaluation.")